In [3]:
import json
from pathlib import Path
import pandas as pd

p = Path(r"/pre processamento/transformation/bronze/Magalu/10-02-2026/busca_tv.json")
data = json.loads(p.read_text(encoding="utf-8"))

if isinstance(data, dict):
    for k in ("items", "data", "results"):
        if k in data and isinstance(data[k], list):
            data = data[k]
            break

df = pd.DataFrame(data)
print(df.columns.tolist())
print(df.head(1).to_dict(orient="records"))


['name', 'category', 'brand', 'price', 'product_url', 'rating', 'in_stock', 'extracted_at']
[{'name': 'Smart TV 43" Philips Full HD DLED 43PFG6918/78 60Hz Google TV Quad Core Google Assistente 3 HDM', 'category': 'tv', 'brand': 'Philips', 'price': 1499.0, 'product_url': 'https://www.magazineluiza.com.br/smart-tv-43-philips-full-hd-dled-43pfg6918-78-60hz-google-tv-quad-core-google-assistente-3-hdm/p/240247300/et/elit/', 'rating': 4.8, 'in_stock': True, 'extracted_at': '2026-02-10T18:37:07-03:00'}]


In [2]:
import json
import re
import unicodedata
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from hashlib import sha1
from pathlib import Path
import pandas as pd

DADOS NULOS

In [4]:

def read_json_list(path: Path):
    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, dict):
        for k in ("items", "data", "results"):
            if k in data and isinstance(data[k], list):
                return data[k]
        raise ValueError(f"JSON inesperado (dict) em {path}")
    if not isinstance(data, list):
        raise ValueError(f"JSON inesperado (não é list) em {path}")
    return data


def summarize_nulls_by_source(bronze_root="bronze"):
    bronze_root = Path(bronze_root)
    stats = {
        "magalu": {"arquivos": 0, "linhas": 0, "nulos_total": 0, "vazios_total": 0},
        "mercado_livre": {"arquivos": 0, "linhas": 0, "nulos_total": 0, "vazios_total": 0},
    }
    for json_path in bronze_root.rglob("*.json"):
        p = str(json_path).lower()
        if "magalu" in p:
            source = "magalu"
        elif "mercado" in p:
            source = "mercado_livre"
        else:
            continue
        rows = read_json_list(json_path)
        df = pd.DataFrame(rows)
        nulos = int(df.isna().sum().sum())
        vazios = 0
        for col in df.columns:
            if df[col].dtype == "object":
                vazios += int(df[col].astype("string").str.strip().eq("").sum())
        stats[source]["arquivos"] += 1
        stats[source]["linhas"] += len(df)
        stats[source]["nulos_total"] += nulos
        stats[source]["vazios_total"] += vazios

    for source, s in stats.items():
        total_celulas = None
        print(f"\n=== {source.upper()} ===")
        print(f"Arquivos analisados: {s['arquivos']}")
        print(f"Linhas (registros) analisadas: {s['linhas']}")
        print(f"Valores NULOS (None/NaN): {s['nulos_total']}")
        print(f"Valores VAZIOS (texto ''): {s['vazios_total']}")
        print(f"Total ausentes (nulos + vazios): {s['nulos_total'] + s['vazios_total']}")

    return stats

In [5]:
summarize_nulls_by_source(bronze_root="bronze")


=== MAGALU ===
Arquivos analisados: 24
Linhas (registros) analisadas: 2471
Valores NULOS (None/NaN): 459
Valores VAZIOS (texto ''): 0
Total ausentes (nulos + vazios): 459

=== MERCADO_LIVRE ===
Arquivos analisados: 18
Linhas (registros) analisadas: 2158
Valores NULOS (None/NaN): 4928
Valores VAZIOS (texto ''): 0
Total ausentes (nulos + vazios): 4928


{'magalu': {'arquivos': 24,
  'linhas': 2471,
  'nulos_total': 459,
  'vazios_total': 0},
 'mercado_livre': {'arquivos': 18,
  'linhas': 2158,
  'nulos_total': 4928,
  'vazios_total': 0}}

tratamento


In [6]:
def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9\s]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_url(url: str) -> str:
    if not url:
        return ""
    return str(url).split("?")[0].strip()

def parse_datetime(dt: str):
    if not dt:
        return None
    dt = str(dt).strip()
    try:
        d = datetime.fromisoformat(dt)
        if d.tzinfo is not None:
            d = d.astimezone(timezone.utc).replace(tzinfo=None)
        return d
    except ValueError:
        pass
    return datetime.strptime(dt, "%Y-%m-%d %H:%M:%S")


def parse_price_br(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        s = str(value)
        if re.match(r"^\d+\.\d{3}$", s):
            return Decimal(s.replace(".", "")).quantize(Decimal("0.01"))
        try:
            return Decimal(str(value)).quantize(Decimal("0.01"))
        except InvalidOperation:
            return None
    s = str(value).strip()
    s = re.sub(r"[^\d,\.]", "", s)
    if "," in s:
        s = s.replace(".", "").replace(",", ".")
    else:
        if re.match(r"^\d+\.\d{3}$", s):
            s = s.replace(".", "")
    try:
        return Decimal(s).quantize(Decimal("0.01"))
    except InvalidOperation:
        return None

def build_product_key(source: str, title_norm: str, price: Decimal, category_norm: str) -> str:
    base = f"{source}|{title_norm}|{price}|{category_norm}"
    return sha1(base.encode("utf-8")).hexdigest()

def read_json_list(path: Path):
    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, dict):
        for k in ("items", "data", "results"):
            if k in data and isinstance(data[k], list):
                return data[k]
        raise ValueError(f"JSON inesperado (dict) em {path}")
    if not isinstance(data, list):
        raise ValueError(f"JSON inesperado (não é list) em {path}")
    return data



In [7]:
def normalize_mercado_livre(df_raw: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "source": "mercado_livre",
        "collected_at": df_raw.get("SCRAPY_DATETIME").map(parse_datetime),
        "title_raw": df_raw.get("TITLE"),
        "price": df_raw.get("PRICE").map(parse_price_br),
        "url": df_raw.get("URL").map(normalize_url),
        "source_url": df_raw.get("SOURCE_URL"),
        "category_raw": df_raw.get("CATEGORY"),
        "rating": df_raw.get("RATING"),
        "in_stock": df_raw.get("IN_STOCK"),
        "availability": df_raw.get("AVAILABILITY"),
    })

def normalize_magalu(df_raw: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "source": "magalu",
        "collected_at": df_raw["extracted_at"].map(parse_datetime),
        "title_raw": df_raw["name"],
        "price": df_raw["price"].map(parse_price_br),
        "url": df_raw["product_url"].map(normalize_url),
        "source_url": None,                 
        "category_raw": df_raw["category"],
        "rating": df_raw["rating"],
        "in_stock": df_raw["in_stock"].fillna(False).astype(bool),
        "availability": None,               
        "brand_raw": df_raw["brand"],       
    })

def finalize_silver(df: pd.DataFrame) -> pd.DataFrame:
    df["collected_at"] = pd.to_datetime(df["collected_at"], errors="coerce")
    df["title_norm"] = df["title_raw"].map(normalize_text)
    df["category_norm"] = df["category_raw"].fillna("").map(normalize_text)
    df["in_stock"] = df["in_stock"].fillna(False).astype(bool)
    df["product_key"] = df.apply(
        lambda r: build_product_key(r["source"], r["title_norm"], r["price"], r["category_norm"]),
        axis=1
    )
    df = df.sort_values(["collected_at"], ascending=False)
    df = df.drop_duplicates(subset=["source", "url"], keep="first")
    df = df[df["price"].notna() & (df["url"] != "")]
    return df.reset_index(drop=True)

def save_df(df: pd.DataFrame, out_path: Path):
    try:
        out_path_parquet = out_path.with_suffix(".parquet")
        df.to_parquet(out_path_parquet, index=False)
        return out_path_parquet
    except Exception as e:
        out_path_csv = out_path.with_suffix(".csv")
        df.to_csv(out_path_csv, index=False, encoding="utf-8-sig")
        return out_path_csv

def process_all_bronze(bronze_root="bronze", silver_root="silver"):
    bronze_root = Path(bronze_root)
    silver_root = Path(silver_root)
    silver_root.mkdir(parents=True, exist_ok=True)
    if not bronze_root.exists():
        raise FileNotFoundError(f"Não achei bronze_root: {bronze_root.resolve()}")
    all_silver = []

    for source_dir in bronze_root.iterdir():
        if not source_dir.is_dir():
            continue
        name = source_dir.name.lower()
        if "magalu" in name:
            normalizer = normalize_magalu
        elif "mercado" in name:
            normalizer = normalize_mercado_livre
        else:
            print(f"[SKIP] Fonte não reconhecida: {source_dir.name}")
            continue
        for json_path in source_dir.rglob("*.json"):
            try:
                rows = read_json_list(json_path)
                df_raw = pd.DataFrame(rows)
                df_std = normalizer(df_raw)
                df_silver = finalize_silver(df_std)
                rel = json_path.relative_to(bronze_root)
                out_dir = (silver_root / rel.parent)
                out_dir.mkdir(parents=True, exist_ok=True)
                out_base = out_dir / json_path.stem
                out_saved = save_df(df_silver, out_base)
                all_silver.append(df_silver)
                print(f"[OK] {json_path} -> {out_saved} ({len(df_silver)} linhas)")
            except Exception as e:
                print(f"[ERRO] Falhou em {json_path}: {e}")

    if all_silver:
        df_all = pd.concat(all_silver, ignore_index=True)
        out_saved = save_df(df_all, silver_root / "silver_all")
        print(f"[OK] Consolidado: {len(df_all)} linhas -> {out_saved}")



In [8]:
process_all_bronze(
    bronze_root="bronze",
)


[OK] bronze/Mercado livre/12-02-2026/mercado_livre_fone_20260212.json -> silver/Mercado livre/12-02-2026/mercado_livre_fone_20260212.csv (96 linhas)
[OK] bronze/Mercado livre/12-02-2026/mercado_livre_smartwatch_20260212.json -> silver/Mercado livre/12-02-2026/mercado_livre_smartwatch_20260212.csv (92 linhas)
[OK] bronze/Mercado livre/12-02-2026/mercado_livre_celular_20260212.json -> silver/Mercado livre/12-02-2026/mercado_livre_celular_20260212.csv (97 linhas)
[OK] bronze/Mercado livre/12-02-2026/mercado_livre_tv_20260212.json -> silver/Mercado livre/12-02-2026/mercado_livre_tv_20260212.csv (97 linhas)
[OK] bronze/Mercado livre/12-02-2026/mercado_livre_notebook_20260212.json -> silver/Mercado livre/12-02-2026/mercado_livre_notebook_20260212.csv (97 linhas)
[OK] bronze/Mercado livre/12-02-2026/mercado_livre_caixa de som_20260212.json -> silver/Mercado livre/12-02-2026/mercado_livre_caixa de som_20260212.csv (97 linhas)
[OK] bronze/Mercado livre/10-02-2026/mercado_livre_caixa de som_2026

onde se concentra mais os dados nulos

In [9]:
def read_json_list(path: Path):
    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, dict):
        for k in ("items", "data", "results"):
            if k in data and isinstance(data[k], list):
                return data[k]
        raise ValueError(f"JSON inesperado (dict) em {path}")
    if not isinstance(data, list):
        raise ValueError(f"JSON inesperado (não é list) em {path}")
    return data


def load_source_df(bronze_root: str, source: str) -> pd.DataFrame:
    bronze_root = Path(bronze_root)
    frames = []
    for p in bronze_root.rglob("*.json"):
        pl = str(p).lower()
        if source == "magalu" and "magalu" not in pl:
            continue
        if source == "mercado_livre" and "mercado" not in pl:
            continue
        rows = read_json_list(p)
        frames.append(pd.DataFrame(rows))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def missing_analysis(df: pd.DataFrame, top_n=5) -> dict:
    if df.empty:
        return {
            "linhas": 0,
            "colunas": 0,
            "celulas": 0,
            "nulos_total": 0,
            "nulos_percent": 0.0,
            "linhas_com_nulo": 0,
            "linhas_com_nulo_percent": 0.0,
            "top_colunas_nulos": pd.DataFrame(columns=["coluna", "nulos", "percent"])
        }
    linhas, colunas = df.shape
    celulas = linhas * colunas
    nulos_por_coluna = df.isna().sum().sort_values(ascending=False)
    nulos_total = int(nulos_por_coluna.sum())
    nulos_percent = round((nulos_total / celulas) * 100, 2) if celulas else 0.0
    linhas_com_nulo = int(df.isna().any(axis=1).sum())
    linhas_com_nulo_percent = round((linhas_com_nulo / linhas) * 100, 2) if linhas else 0.0
    top = nulos_por_coluna.head(top_n).reset_index()
    top.columns = ["coluna", "nulos"]
    top["percent"] = (top["nulos"] / linhas * 100).round(2)

    return {
        "linhas": linhas,
        "colunas": colunas,
        "celulas": celulas,
        "nulos_total": nulos_total,
        "nulos_percent": nulos_percent,
        "linhas_com_nulo": linhas_com_nulo,
        "linhas_com_nulo_percent": linhas_com_nulo_percent,
        "top_colunas_nulos": top
    }

def run_missing_report(bronze_root="bronze", top_n=5):
    for source in ("magalu", "mercado_livre"):
        df = load_source_df(bronze_root, source)
        result = missing_analysis(df, top_n=top_n)

        print(f"\n=== {source.upper()} ===")
        print(f"Linhas: {result['linhas']} | Colunas: {result['colunas']} | Células: {result['celulas']}")
        print(f"Nulos (células): {result['nulos_total']}  ({result['nulos_percent']}%)")
        print(f"Linhas com >=1 nulo: {result['linhas_com_nulo']}  ({result['linhas_com_nulo_percent']}%)")
        print("\nTop colunas com mais nulos:")
        print(result["top_colunas_nulos"].to_string(index=False))



In [10]:
run_missing_report(bronze_root="bronze", top_n=5)


=== MAGALU ===
Linhas: 2471 | Colunas: 8 | Células: 19768
Nulos (células): 459  (2.32%)
Linhas com >=1 nulo: 433  (17.52%)

Top colunas com mais nulos:
  coluna  nulos  percent
  rating    384    15.54
   brand     75     3.04
    name      0     0.00
category      0     0.00
   price      0     0.00

=== MERCADO_LIVRE ===
Linhas: 2158 | Colunas: 9 | Células: 19422
Nulos (células): 4928  (25.37%)
Linhas com >=1 nulo: 1674  (77.57%)

Top colunas com mais nulos:
      coluna  nulos  percent
      RATING   1674    77.57
    IN_STOCK   1627    75.39
AVAILABILITY   1627    75.39
       TITLE      0     0.00
       PRICE      0     0.00
